[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_21/2_sql.ipynb)

# Day 21: SQL (mock exam)

**Mock exam rules (30 minutes, one timer for the whole notebook).**

- Start one 30 minute timer, then work through Q1 to Q6 in any order. That is about 5 minutes a query: exam pace.
- Do not open any hint or solution until the timer rings. When it rings, stop, then grade yourself with the solutions.
- Score: 1 point for a correct result, 0.5 if the logic is right but a detail is wrong (rounding, a missing filter).
  5 or more is exam ready. Every question you missed goes into the mistake log.
- Topics are mixed on purpose (dedup, funnels, retention, self-joins, ratios, streaks). Part of the test is spotting which one.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (32 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Clean the raw event stream | hard | 5 |
| Q2 | Ordered checkout funnel by platform | hard | 6 |
| Q3 | Day 1 and day 7 retention by signup week (review) | hard | 5 |
| Q4 | Genres bought together (with lift) | hard | 6 |
| Q5 | Which acquisition channel brings valuable users? | hard | 5 |
| Q6 | Longest daily streak | hard | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Clean the raw event stream

*hard, about 5 min*

`app_events_raw` is the log as the apps send it. Two kinds of junk are in it:
- **retries**: an exact copy of an event (same `user_id`, `event_time`, `event_name`), only `raw_id` and `received_at` differ;
- **double taps**: the same user fires the same event again 1 to 3 seconds later.

Rule: drop exact copies, then drop an event if the same user had the same `event_name` **5 seconds or less** before it.
Return one row per `event_name`: `raw_rows`, `clean_rows`, `removed`, ordered by `raw_rows` descending.

Example: user 7 sends `view_item` at 10:00:00 (twice, a retry) and again at 10:00:02 (a tap). Clean result: 1 row.

*Follow-up (say it out loud):* how would you check that your rule did not delete real events?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'copies' and 'within N seconds of the previous one'. Pattern: dedup-cleaning. SELECT DISTINCT for exact copies, then LAG over (user, event) ordered by time for near copies.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE `exact`: SELECT DISTINCT user_id, event_time, event_name.
2. CTE `flagged`: add `LAG(event_time) OVER (PARTITION BY user_id, event_name ORDER BY event_time)`.
3. Keep rows where the previous time is NULL or more than 5 seconds earlier (`(julianday(t) - julianday(prev)) * 86400 > 5`).
4. Count raw and clean rows per event and join them.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH exact AS (
  SELECT DISTINCT user_id, event_time, event_name
  FROM app_events_raw
), flagged AS (
  SELECT *,
    LAG(event_time) OVER (PARTITION BY user_id, event_name ORDER BY event_time) AS prev_time
  FROM exact
), clean AS (
  SELECT * FROM flagged
  WHERE prev_time IS NULL
     OR (julianday(event_time) - julianday(prev_time)) * 86400 > 5
)
SELECT r.event_name, r.raw_rows, c.clean_rows, r.raw_rows - c.clean_rows AS removed
FROM (SELECT event_name, COUNT(*) AS raw_rows FROM app_events_raw GROUP BY event_name) r
JOIN (SELECT event_name, COUNT(*) AS clean_rows FROM clean GROUP BY event_name) c
  ON c.event_name = r.event_name
ORDER BY r.raw_rows DESC
```

Result:

| event_name | raw_rows | clean_rows | removed |
|---|---|---|---|
| view_item | 11587 | 11003 | 584 |
| add_to_cart | 4859 | 4638 | 221 |
| checkout | 2570 | 2439 | 131 |
| purchase | 1066 | 1014 | 52 |

**Why:** Retries are identical on the business columns, so DISTINCT on those columns (not on `raw_id`) removes them. Double taps differ by a few seconds, so only a comparison with the previous event of the same user and type finds them: that is LAG with the right PARTITION BY. Follow-up answer: the clean counts (11,003 views, 4,638 carts, 2,439 checkouts, 1,014 purchases) match the trusted `app_events` table exactly, which is the check you describe: compare with a trusted source, and look at the gap distribution (real repeats are 15 s or more apart).

**Complexity:** One sort per (user, event) partition: O(n log n).

**Common mistakes:** DISTINCT on all columns (raw_id makes every row unique, nothing is removed). Comparing with the previous event of ANY type (a view then a cart 2 s later is real). In PostgreSQL write `event_time - prev_time <= interval '5 seconds'`; julianday is SQLite only.

**Learn more:** [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead)

</details>

---
### Q2. Ordered checkout funnel by platform

*hard, about 6 min*

Users who signed up in **January 2024** (`app_users`). For each user look at their first 7 days
(signup day plus the next 6, i.e. `event_time < signup_date + 7 days`). The funnel is
`view_item -> add_to_cart -> checkout -> purchase`, and **order matters**: a step only counts if it happens after the
user's first time at the previous step (an `add_to_cart` before any `view_item` does not count).

Return per `platform`: `users`, `viewed`, `carted`, `checked_out`, `purchased`, `pct_purchase` (purchased / users, 1 decimal).

Example: a user with cart 09:00, view 09:05, purchase 09:20 and no checkout counts as viewed only.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'order matters' and 'first time at the previous step'. Pattern: funnels with chained CTEs: each step joins events that happen after the previous step's time.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE `u`: January signups.
2. `s1`: first view per user inside the window.
3. `s2`: first cart with `event_time > s1.t`, joined from s1 (so only viewers can reach it). Same for s3, s4.
4. `u LEFT JOIN s1 ... s4`, GROUP BY platform, COUNT(sN.user_id).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH u AS (
  SELECT user_id, platform, signup_date, date(signup_date, '+7 days') AS window_end
  FROM app_users
  WHERE signup_date >= '2024-01-01' AND signup_date < '2024-02-01'
), s1 AS (
  SELECT u.user_id, MIN(e.event_time) AS t
  FROM u JOIN app_events e ON e.user_id = u.user_id
  WHERE e.event_name = 'view_item' AND e.event_time < u.window_end
  GROUP BY u.user_id
), s2 AS (
  SELECT s1.user_id, MIN(e.event_time) AS t
  FROM s1 JOIN u ON u.user_id = s1.user_id
  JOIN app_events e ON e.user_id = s1.user_id
  WHERE e.event_name = 'add_to_cart' AND e.event_time > s1.t AND e.event_time < u.window_end
  GROUP BY s1.user_id
), s3 AS (
  SELECT s2.user_id, MIN(e.event_time) AS t
  FROM s2 JOIN u ON u.user_id = s2.user_id
  JOIN app_events e ON e.user_id = s2.user_id
  WHERE e.event_name = 'checkout' AND e.event_time > s2.t AND e.event_time < u.window_end
  GROUP BY s2.user_id
), s4 AS (
  SELECT s3.user_id, MIN(e.event_time) AS t
  FROM s3 JOIN u ON u.user_id = s3.user_id
  JOIN app_events e ON e.user_id = s3.user_id
  WHERE e.event_name = 'purchase' AND e.event_time > s3.t AND e.event_time < u.window_end
  GROUP BY s3.user_id
)
SELECT u.platform, COUNT(*) AS users,
  COUNT(s1.user_id) AS viewed, COUNT(s2.user_id) AS carted,
  COUNT(s3.user_id) AS checked_out, COUNT(s4.user_id) AS purchased,
  ROUND(100.0 * COUNT(s4.user_id) / COUNT(*), 1) AS pct_purchase
FROM u
LEFT JOIN s1 ON s1.user_id = u.user_id
LEFT JOIN s2 ON s2.user_id = u.user_id
LEFT JOIN s3 ON s3.user_id = u.user_id
LEFT JOIN s4 ON s4.user_id = u.user_id
GROUP BY u.platform
ORDER BY u.platform
```

Result:

| platform | users | viewed | carted | checked_out | purchased | pct_purchase |
|---|---|---|---|---|---|---|
| android | 360 | 306 | 193 | 119 | 56 | 15.6 |
| ios | 311 | 269 | 172 | 109 | 53 | 17.0 |
| web | 108 | 90 | 50 | 30 | 12 | 11.1 |

**Why:** Each step CTE starts from the users who passed the previous step and only looks at later events, so the funnel is monotone (viewed >= carted >= checked_out >= purchased) by construction. LEFT JOIN from `u` keeps users who never viewed, so the denominator is all signups. iOS converts best (17.0%), web worst (11.1%).

**Complexity:** Each step is a join on user_id plus a MIN: about O(events) with an index on (user_id, event_name, event_time).

**Common mistakes:** Counting users who did each event at any time (not ordered, not monotone). Using `MIN(CASE ...)` per step and comparing first times: the first cart can be before the first view while a later cart is valid. Forgetting the 7 day window or using `<=` on a date string against a timestamp.

**Learn more:** [sql-funnels](https://mahsa-agz.github.io/ds-handbook/#sql-funnels), [sql-cte](https://mahsa-agz.github.io/ds-handbook/#sql-cte)

</details>

---
### Q3. Day 1 and day 7 retention by signup week (review)

*hard, about 5 min*

Group users by signup week (weeks start on **Monday**). For each cohort week return `users`,
`d1_pct` (share active exactly 1 day after signup) and `d7_pct` (active exactly 7 days after signup), in percent with 1
decimal, using `daily_activity`. Order by cohort week.

Example: signup 2024-01-03 (a Wednesday) belongs to cohort week 2024-01-01; day 7 is 2024-01-10.

*Follow-up:* the last cohort looks worse. Is it a real drop?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'cohort', 'active N days after signup'. Pattern: retention: cohort CTE + LEFT JOIN activity on the exact day.

</details>

<details><summary><b>Hint 2</b></summary>

1. Week start in SQLite: `date(d, 'weekday 0', '-6 days')` (next Sunday, back 6 days = Monday).
2. LEFT JOIN `daily_activity` twice, with `activity_date = date(signup_date, '+1 day')` and `'+7 days'`.
3. COUNT(d1.user_id) / COUNT(*) per cohort, times 100.0.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH cohort AS (
  SELECT user_id, signup_date,
    date(signup_date, 'weekday 0', '-6 days') AS cohort_week
  FROM app_users
)
SELECT c.cohort_week, COUNT(*) AS users,
  ROUND(100.0 * COUNT(d1.user_id) / COUNT(*), 1) AS d1_pct,
  ROUND(100.0 * COUNT(d7.user_id) / COUNT(*), 1) AS d7_pct
FROM cohort c
LEFT JOIN daily_activity d1
  ON d1.user_id = c.user_id AND d1.activity_date = date(c.signup_date, '+1 day')
LEFT JOIN daily_activity d7
  ON d7.user_id = c.user_id AND d7.activity_date = date(c.signup_date, '+7 days')
GROUP BY c.cohort_week
ORDER BY c.cohort_week
```

Result:

| cohort_week | users | d1_pct | d7_pct |
|---|---|---|---|
| 2024-01-01 | 174 | 45.4 | 29.9 |
| 2024-01-08 | 179 | 41.9 | 29.1 |
| 2024-01-15 | 170 | 49.4 | 26.5 |
| 2024-01-22 | 181 | 45.3 | 39.8 |
| 2024-01-29 | 175 | 50.9 | 28.6 |
| 2024-02-05 | 168 | 44.6 | 28.0 |
| 2024-02-12 | 200 | 41.5 | 28.5 |
| 2024-02-19 | 156 | 42.3 | 30.8 |
| 2024-02-26 | 97 | 39.2 | 22.7 |

**Why:** The day condition sits in the JOIN, not in WHERE, so retained-or-not users all stay in the denominator. `daily_activity` has one row per user and day, so the joins cannot multiply rows. Follow-up: the last cohort (2024-02-26, 97 users, d7 22.7%) is a partial week (Monday to Thursday, signups stop on 2024-02-29) and small, so it is noisy; also check day of week effects before calling it a drop.

**Complexity:** Two equality joins on (user_id, date): O(n) with an index.

**Common mistakes:** Putting `d7.activity_date = ...` in WHERE (turns the LEFT JOIN into an inner join: 100% retention). Counting activity on day 1 to 7 (that is rolling retention, a different metric). PostgreSQL: `date_trunc('week', signup_date)` gives the Monday.

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q4. Genres bought together (with lift)

*hard, about 6 min*

Chinook (real). Two genres are "bought together" when both appear on the same invoice. For every pair of
genres (each pair once) with at least 10 shared invoices, return `genre_1`, `genre_2`, `both_inv` and
`lift = P(both) / (P(g1) * P(g2))`, where P is a share of all invoices. Show the top 5 by lift (ties: more shared
invoices first), lift rounded to 2 decimals.

Example: 412 invoices, Rock on 100, Jazz on 50, both on 20: lift = (20/412) / ((100/412) * (50/412)) = 1.65.

*Follow-up:* why not rank by `both_inv` alone?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'pairs on the same invoice'. Pattern: self-join of (invoice, genre) on invoice_id with `a.genre < b.genre`, then a ratio with per-genre counts.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE `inv_genre`: DISTINCT (InvoiceId, GenreId) from InvoiceLine join Track.
2. `pairs`: self-join on InvoiceId with `a.GenreId < b.GenreId`, COUNT(*).
3. `single`: invoices per genre; `n`: total invoices.
4. lift = `1.0 * both * total / (inv1 * inv2)`; filter both >= 10; ORDER BY lift DESC.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH inv_genre AS (
  SELECT DISTINCT il.InvoiceId, t.GenreId
  FROM InvoiceLine il JOIN Track t ON t.TrackId = il.TrackId
), n AS (
  SELECT COUNT(DISTINCT InvoiceId) AS total FROM inv_genre
), single AS (
  SELECT GenreId, COUNT(*) AS inv FROM inv_genre GROUP BY GenreId
), pairs AS (
  SELECT a.GenreId AS g1, b.GenreId AS g2, COUNT(*) AS both_inv
  FROM inv_genre a
  JOIN inv_genre b ON a.InvoiceId = b.InvoiceId AND a.GenreId < b.GenreId
  GROUP BY a.GenreId, b.GenreId
)
SELECT ga.Name AS genre_1, gb.Name AS genre_2, p.both_inv,
  ROUND(1.0 * p.both_inv * n.total / (s1.inv * s2.inv), 2) AS lift
FROM pairs p
CROSS JOIN n
JOIN single s1 ON s1.GenreId = p.g1
JOIN single s2 ON s2.GenreId = p.g2
JOIN Genre ga ON ga.GenreId = p.g1
JOIN Genre gb ON gb.GenreId = p.g2
WHERE p.both_inv >= 10
ORDER BY lift DESC, p.both_inv DESC
LIMIT 5
```

Result:

| genre_1 | genre_2 | both_inv | lift |
|---|---|---|---|
| TV Shows | Drama | 10 | 15.49 |
| Rock | Soundtrack | 12 | 1.91 |
| Rock | Reggae | 11 | 1.61 |
| Jazz | Metal | 15 | 1.57 |
| Jazz | Latin | 17 | 1.46 |

**Why:** DISTINCT first makes one row per (invoice, genre), so an invoice with 5 Rock tracks counts once. `a.GenreId < b.GenreId` keeps each pair once and drops self pairs. Follow-up: raw co-occurrence is dominated by popular genres (Rock appears with everything); lift divides by what chance alone predicts. TV Shows + Drama has lift 15.49 (people buy TV episodes together); Rock + Alternative & Punk has the most shared invoices (54) but a lift of only 1.11, so it is mostly popularity. The minimum count guards against tiny noisy pairs.

**Complexity:** Self-join per invoice: O(sum of k^2) where k = genres per invoice (small).

**Common mistakes:** Self-joining InvoiceLine without DISTINCT (counts track pairs, not invoices). Using `<>` instead of `<` (each pair twice). Integer division in the lift (multiply by 1.0 first).

**Learn more:** [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q5. Which acquisition channel brings valuable users?

*hard, about 5 min*

Per `channel` in `app_users` return:
- `users`: signed-up users;
- `buyer_pct`: % of users with at least one **completed** purchase;
- `net_rev_per_user`: completed revenue divided by ALL users of the channel (2 decimals);
- `refund_pct`: refunded orders / all orders of the channel (1 decimal; NULL if no orders).

Order by `net_rev_per_user` descending. Users with no purchase must count in the denominators.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: per-group ratios with different denominators. Pattern: ratios with LEFT JOIN + conditional aggregation (CASE inside SUM / COUNT DISTINCT), and NULLIF for a safe division.

</details>

<details><summary><b>Hint 2</b></summary>

1. `app_users u LEFT JOIN purchases p`.
2. users = COUNT(DISTINCT u.user_id) (the join repeats users).
3. buyers = COUNT(DISTINCT CASE WHEN p.status = 'completed' THEN u.user_id END).
4. revenue = SUM(CASE ... THEN amount ELSE 0 END); refunds over `NULLIF(COUNT(p.order_id), 0)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT u.channel,
  COUNT(DISTINCT u.user_id) AS users,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN p.status = 'completed' THEN u.user_id END)
        / COUNT(DISTINCT u.user_id), 1) AS buyer_pct,
  ROUND(SUM(CASE WHEN p.status = 'completed' THEN p.amount ELSE 0 END)
        / COUNT(DISTINCT u.user_id), 2) AS net_rev_per_user,
  ROUND(100.0 * SUM(CASE WHEN p.status = 'refunded' THEN 1 ELSE 0 END)
        / NULLIF(COUNT(p.order_id), 0), 1) AS refund_pct
FROM app_users u
LEFT JOIN purchases p ON p.user_id = u.user_id
GROUP BY u.channel
ORDER BY net_rev_per_user DESC
```

Result:

| channel | users | buyer_pct | net_rev_per_user | refund_pct |
|---|---|---|---|---|
| referral | 227 | 26.4 | 29.08 | 7.6 |
| organic | 545 | 28.3 | 28.5 | 6.0 |
| search | 303 | 26.7 | 22.28 | 5.9 |
| paid_social | 425 | 25.4 | 20.97 | 7.5 |

**Why:** After the LEFT JOIN a user appears once per order, so user counts need DISTINCT while order counts must not. Referral users bring the most net revenue per signup (29.08) and paid social the least (20.97), even though buyer rates are close (25.4% to 28.3%), so the gap is in order value, not conversion.

**Complexity:** One join and one GROUP BY: O(users + orders).

**Common mistakes:** Revenue per BUYER instead of per user (changes the ranking). COUNT(*) as the user count (counts orders). `100 * a / b` with integers. Including refunded amounts in revenue.

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when)

</details>

---
### Q6. Longest daily streak

*hard, about 5 min*

Using `daily_activity` (one row per user per active day), find each user's **longest run of consecutive
active days**. Return the top 5 users: `user_id`, `start_date`, `end_date`, `days`, ordered by `days` descending then
`user_id`. If a user has two runs of the same length, show the earlier one.

Example: active on Jan 3, 4, 5, 7, 8: runs are 3 days (Jan 3 to 5) and 2 days; the longest is 3.

*Follow-up:* how many users ever had a streak of 7 days or more? (Change one line.)

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'consecutive days'. Pattern: gaps and islands: day number minus ROW_NUMBER is constant inside a run.

</details>

<details><summary><b>Hint 2</b></summary>

1. `julianday(activity_date) - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date)` = group key.
2. GROUP BY user_id, key: MIN date, MAX date, COUNT.
3. ROW_NUMBER over runs per user ordered by days DESC, start_date; keep rn = 1; top 5.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH g AS (
  SELECT user_id, activity_date,
    julianday(activity_date)
      - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date) AS grp
  FROM daily_activity
), streaks AS (
  SELECT user_id, MIN(activity_date) AS start_date, MAX(activity_date) AS end_date,
         COUNT(*) AS days
  FROM g
  GROUP BY user_id, grp
), ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY days DESC, start_date) AS rn
  FROM streaks
)
SELECT user_id, start_date, end_date, days
FROM ranked
WHERE rn = 1
ORDER BY days DESC, user_id
LIMIT 5
```

Result:

| user_id | start_date | end_date | days |
|---|---|---|---|
| 259 | 2024-01-11 | 2024-03-12 | 62 |
| 166 | 2024-01-27 | 2024-03-27 | 61 |
| 698 | 2024-01-28 | 2024-03-21 | 54 |
| 935 | 2024-02-07 | 2024-03-31 | 54 |
| 522 | 2024-02-11 | 2024-03-30 | 49 |

**Why:** Inside a run the date grows by 1 and the row number grows by 1, so their difference stays constant; a gap changes it. That turns 'consecutive' into a plain GROUP BY. Follow-up: replace the final SELECT with `SELECT COUNT(DISTINCT user_id) FROM streaks WHERE days >= 7`: 384 users.

**Complexity:** One window sort per user: O(n log n).

**Common mistakes:** Using ROW_NUMBER when a user can have two rows on the same day (then dedup first or use DENSE_RANK). Subtracting a row number from a date string (convert to a day number first). In PostgreSQL: `activity_date - ROW_NUMBER() OVER (...) * INTERVAL '1 day'`.

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_21/3_stats.ipynb)